# Preprocessing Fix — Unit Normalization
Insert this notebook's cells into `02_Preprocessing.ipynb` **after Step 3 (LOINC standardization)**  
and **before Step 5 (Select top tests)**.

The issue: different labs report the same test in different units.  
e.g., Platelet Count in ×10³/µL (value ~150–400) vs cells/µL (value ~150,000–400,000).  
After pivoting and averaging, these get mixed and produce nonsense values.

## Step A — Inspect Unit Distributions for Problematic Features

In [ ]:
# These are the LOINC codes confirmed to have unit mixing issues
problematic_loincs = {
    "26515-7": "Platelet Count",
    "26464-8": "Total WBC Count",
    "26499-4": "Absolute Neutrophil Count",
    "26474-7": "Absolute Lymphocyte Count",
    "26484-6": "Absolute Monocyte Count",
    "26449-9": "Absolute Eosinophil Count",
    "26444-0": "Absolute Basophil Count",
    "51637-7": "PCT (Plateletcrit)",
}

for loinc, name in problematic_loincs.items():
    subset = df_filtered[df_filtered["loinc"] == loinc]
    print(f"\n{'='*60}")
    print(f"{name} ({loinc}) — {len(subset):,} rows")
    print("Unit counts:")
    print(subset["unit"].value_counts().head(10).to_string())
    print("Value stats:")
    print(subset["value"].describe().round(2).to_string())

## Step B — Define Normalization Rules
Based on unit inspection above, we define:
- The **standard unit** for each test  
- A **value threshold** to detect non-standard scale  
- The **conversion factor** to apply

Strategy: use value thresholds as the primary detection method  
(more reliable than unit string matching which is inconsistently formatted).

In [ ]:
# Rules: (loinc, standard_max, conversion_factor, description)
# If value > standard_max → it's on the large scale → multiply by conversion_factor
# standard_max is set between the two scales to cleanly separate them

normalization_rules = {
    # Platelet: ×10³/µL (150-400) vs /µL (150,000-400,000)
    # threshold = 5000: anything above is in /µL → divide by 1000
    "26515-7": {"name": "Platelet Count",               "threshold": 5000,  "factor": 1/1000},

    # Total WBC: ×10³/µL (4-10) vs cells/cumm (4000-10000)
    # threshold = 100: anything above is in cells/µL → divide by 1000
    "26464-8": {"name": "Total WBC Count",              "threshold": 100,   "factor": 1/1000},

    # Absolute Neutrophil: ×10³/µL (2-7) vs cells/µL (2000-7000)
    "26499-4": {"name": "Absolute Neutrophil Count",    "threshold": 100,   "factor": 1/1000},

    # Absolute Lymphocyte: ×10³/µL (1-3) vs cells/µL (1000-3000)
    "26474-7": {"name": "Absolute Lymphocyte Count",    "threshold": 100,   "factor": 1/1000},

    # Absolute Monocyte: ×10³/µL (0.2-1.0) vs cells/µL (200-1000)
    "26484-6": {"name": "Absolute Monocyte Count",      "threshold": 50,    "factor": 1/1000},

    # Absolute Eosinophil: ×10³/µL (0.02-0.5) vs cells/µL (20-500)
    "26449-9": {"name": "Absolute Eosinophil Count",    "threshold": 10,    "factor": 1/1000},

    # Absolute Basophil: ×10³/µL (0.01-0.1) vs cells/µL (10-100)
    "26444-0": {"name": "Absolute Basophil Count",      "threshold": 5,     "factor": 1/1000},
}

# PCT is excluded — two conflicting reference ranges found on site,
# suggesting extraction errors. Will drop it as unreliable.
PCT_LOINC = "51637-7"
print("Normalization rules defined.")
print(f"PCT ({PCT_LOINC}) will be dropped as unreliable.")

## Step C — Apply Unit Normalization

In [ ]:
import copy

df_normalized = df_filtered.copy()
conversion_log = []

for loinc, rule in normalization_rules.items():
    mask_loinc = df_normalized["loinc"] == loinc
    mask_high  = df_normalized["value"] > rule["threshold"]
    mask_both  = mask_loinc & mask_high

    n_converted = mask_both.sum()
    n_total     = mask_loinc.sum()

    if n_converted > 0:
        df_normalized.loc[mask_both, "value"] = (
            df_normalized.loc[mask_both, "value"] * rule["factor"]
        )

    conversion_log.append({
        "test"         : rule["name"],
        "loinc"        : loinc,
        "total_rows"   : n_total,
        "converted"    : n_converted,
        "pct_converted": round(n_converted / n_total * 100, 1) if n_total > 0 else 0
    })

log_df = pd.DataFrame(conversion_log)
print("Conversion summary:")
print(log_df.to_string(index=False))

In [ ]:
# Drop PCT — confirmed unreliable
before = len(df_normalized)
df_normalized = df_normalized[df_normalized["loinc"] != PCT_LOINC]
print(f"Dropped PCT rows: {before - len(df_normalized):,}")
print(f"Remaining: {len(df_normalized):,}")

## Step D — Verify Normalization Worked

In [ ]:
# After normalization, all values for these tests should be in the standard range
print("Post-normalization value ranges (should now be in ×10³/µL scale):\n")
for loinc, rule in normalization_rules.items():
    vals = df_normalized[df_normalized["loinc"] == loinc]["value"]
    if len(vals) == 0:
        continue
    p1, p99 = vals.quantile(0.01), vals.quantile(0.99)
    print(f"{rule['name']:40s}  p1={p1:.3f}  median={vals.median():.3f}  p99={p99:.3f}")

In [ ]:
# Visual check: before vs after for platelet count
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Before (from df_filtered)
before_vals = df_filtered[df_filtered["loinc"] == "26515-7"]["value"].dropna()
before_vals.clip(0, 600000).hist(bins=80, ax=axes[0], color="salmon", edgecolor="white")
axes[0].set_title("Platelet Count — BEFORE normalization")
axes[0].set_xlabel("Value")

# After
after_vals = df_normalized[df_normalized["loinc"] == "26515-7"]["value"].dropna()
after_vals.clip(0, 1000).hist(bins=80, ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Platelet Count — AFTER normalization (×10³/µL)")
axes[1].set_xlabel("Value (×10³/µL)")
axes[1].axvline(150, color="green", linestyle="--", label="Normal lower (150)")
axes[1].axvline(400, color="red", linestyle="--", label="Normal upper (400)")
axes[1].legend()

plt.tight_layout()
plt.savefig("./results/plots/platelet_normalization_check.png", dpi=150)
plt.show()

In [ ]:
# Replace df_filtered with the normalized version for the rest of preprocessing
df_filtered = df_normalized
print("df_filtered updated with normalized units.")
print(f"Shape: {df_filtered.shape}")
print("\nContinue from Step 5 (Select top tests) onwards.")